# 04 — Clean Proximity Data
Loads ECHO PFAS facilities, PWS service area boundaries, PFAS measurements, and demographics.
Merges them into a single analysis-ready dataset saved to `data/cleaned_data/`.

In [ ]:
import os
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path

def find_repo_root(start=Path().resolve()):
    for parent in [start] + list(start.parents):
        if (parent / 'README.md').exists() and (parent / 'data').exists():
            return parent
    raise FileNotFoundError('Could not find repo root.')

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
DATA_IN  = REPO_ROOT / 'data' / '01_pulls'
DATA_OUT = REPO_ROOT / 'data' / 'cleaned_data'
DATA_OUT.mkdir(exist_ok=True)

print('Working directory:', os.getcwd())

## Loading Raw Data
Loading the files saved by `01_data_pull_me_search.ipynb` from `data/01_pulls/`.

In [ ]:
# Load ECHO PFAS-handling facilities
echo_df = pd.read_csv(
    DATA_IN / 'echo_pfas_facilities_nj.csv',
    dtype={'FAC_FIPS_CODE': str}
)
echo_df['Latitude']  = pd.to_numeric(echo_df['Latitude'],  errors='coerce')
echo_df['Longitude'] = pd.to_numeric(echo_df['Longitude'], errors='coerce')
echo_df = echo_df.dropna(subset=['Latitude', 'Longitude'])

print(f'ECHO facilities loaded: {len(echo_df)}')
print(echo_df[['Facility', 'Industry', 'Latitude', 'Longitude']].head())

In [ ]:
# Load PWS service area boundaries
pws_gdf = gpd.read_file(DATA_IN / 'pws_boundaries_nj.geojson')

print(f'PWS boundaries loaded: {len(pws_gdf)}')
print(pws_gdf[['PWSID', 'PWS_Name', 'Population_Served_Count']].head())

In [ ]:
# Load PWS → Census tract crosswalk
xwalk_df = pd.read_csv(
    DATA_IN / 'pws_tract_crosswalk_nj.csv',
    dtype={'GEOID20': str, 'PWSID': str}
)

print(f'Tract crosswalk loaded: {len(xwalk_df)} rows')
print(xwalk_df.head())

In [ ]:
# Load UCMR5 PFAS data and aggregate by PWSID
ucmr_df = pd.read_csv(DATA_IN / 'ucmr5_nj_slim.csv', dtype={'PWSID': str})
ucmr_df['AnalyticalResultValue'] = ucmr_df['AnalyticalResultValue'].fillna(0)

# Keep only PFAS contaminants
pfas_df = ucmr_df[
    ucmr_df['Contaminant'].str.startswith('PF') |
    ucmr_df['Contaminant'].str.contains('HFPO-DA')
].copy()

# Aggregate to one row per PWSID
pfas_by_pws = pfas_df.groupby('PWSID').agg(
    max_pfas         = ('AnalyticalResultValue', 'max'),
    mean_pfas        = ('AnalyticalResultValue', 'mean'),
    total_detections = ('AnalyticalResultValue', lambda x: (x > 0).sum()),
    pws_name         = ('PWSName', 'first')
).reset_index()

pfas_by_pws['exceeds_mcl'] = pfas_by_pws['max_pfas'] > 0.004

print(f'PFAS data: {len(pfas_by_pws)} water systems')
print(pfas_by_pws.head())

In [ ]:
# Load cleaned census demographics (from notebook 02)
census_df = pd.read_csv(
    DATA_OUT / 'final_clean_census.csv',
    dtype={'ZIPCODE': str}
)

# Load ZIP → PWSID crosswalk to link demographics to water systems
zip_pws = pd.read_csv(
    DATA_IN / 'zipcodes_raw.csv',
    dtype={'PWSID': str, 'ZIPCODE': str}
)
zip_pws['ZIPCODE'] = zip_pws['ZIPCODE'].str.zfill(5)

print(f'Census data: {len(census_df)} ZIPs')
print(f'ZIP-PWS crosswalk: {len(zip_pws)} rows')

## Spatial Join: ECHO Facilities → PWS Service Areas

In [ ]:
# Convert ECHO facilities to GeoDataFrame
echo_gdf = gpd.GeoDataFrame(
    echo_df,
    geometry = gpd.points_from_xy(echo_df['Longitude'], echo_df['Latitude']),
    crs      = 'EPSG:4326'
)

# Match CRS
pws_gdf = pws_gdf.to_crs('EPSG:4326')

# Spatial join: which PWS service area does each facility fall in?
facilities_in_pws = gpd.sjoin(
    echo_gdf,
    pws_gdf[['PWSID', 'PWS_Name', 'geometry']],
    how       = 'left',
    predicate = 'within'
)

matched = facilities_in_pws['PWSID'].notna().sum()
print(f'Facilities matched to a PWS service area: {matched} of {len(echo_gdf)}')
print(facilities_in_pws[['Facility', 'Industry', 'PWSID']].head())

In [ ]:
# Aggregate facilities per PWSID: total count and count by industry type
fac_matched = facilities_in_pws[facilities_in_pws['PWSID'].notna()].copy()

# Total facility count per PWS
fac_count = fac_matched.groupby('PWSID').agg(
    total_facilities = ('Facility', 'count'),
).reset_index()

# Count by industry type (pivot)
industry_counts = (
    fac_matched.groupby(['PWSID', 'Industry'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
# Clean up column names
industry_counts.columns = [
    'PWSID' if c == 'PWSID' else f'n_{c.lower().replace(" ", "_")}'
    for c in industry_counts.columns
]

fac_summary = fac_count.merge(industry_counts, on='PWSID', how='left')

print(f'PWS with at least one facility: {len(fac_summary)}')
print(fac_summary.head())

## Merging Everything Together

In [ ]:
# Start with PWS boundaries as the base
proximity_df = pws_gdf[['PWSID', 'PWS_Name', 'Population_Served_Count', 'geometry']].copy()

# Join PFAS levels
proximity_df = proximity_df.merge(pfas_by_pws, on='PWSID', how='left')

# Join facility counts (fill 0 for PWS with no facilities)
proximity_df = proximity_df.merge(fac_summary, on='PWSID', how='left')
proximity_df['total_facilities'] = proximity_df['total_facilities'].fillna(0).astype(int)
proximity_df['has_facility'] = proximity_df['total_facilities'] > 0

# Join demographics via ZIP → PWSID crosswalk
zip_demo = zip_pws.merge(census_df, on='ZIPCODE', how='left')
demo_by_pws = zip_demo.groupby('PWSID')[[
    'pct_poc', 'pct_black', 'pct_hispanic', 'pct_nonhispanic_white',
    'hh_median_income', 'total_pop'
]].mean().reset_index()

proximity_df = proximity_df.merge(demo_by_pws, on='PWSID', how='left')

print(f'Final dataset: {len(proximity_df)} water systems')
print(proximity_df.columns.tolist())

## Saving

In [ ]:
# Save GeoDataFrame (with geometry) for mapping in notebook 05
proximity_df.to_file(DATA_OUT / 'proximity_final.geojson', driver='GeoJSON')

# Save flat CSV (without geometry) for stats/charts in notebook 05
proximity_df.drop(columns='geometry').to_csv(
    DATA_OUT / 'proximity_final.csv', index=False
)

# Save the facility-level data with PWS assignments for mapping
facilities_in_pws.drop(columns='geometry').to_csv(
    DATA_OUT / 'echo_facilities_with_pws.csv', index=False
)

print('Saved:')
print('  proximity_final.geojson')
print('  proximity_final.csv')
print('  echo_facilities_with_pws.csv')